In [ ]:
import cv2, torch
from depth_anything_v2.dpt import DepthAnythingV2


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = DepthAnythingV2(
    encoder="vits",
    features=64,
    out_channels=[48, 96, 192, 384]
)

model.load_state_dict(
    torch.load(
        "../checkpoints/depth_anything_v2_vits.pth",
        map_location="cpu"
    )
)

model = model.to(DEVICE).eval()


In [ ]:
from matplotlib import pyplot as plt
import numpy as np

# ---- inference ----
img = cv2.imread("../photos/wood_dish_1.png")
img = cv2.imread("../photos/ok_light_2.png")
# img = cv2.imread("../photos/good_light_1.png")
depth = model.infer_image(img)   # H x W float32

# ---- depth visualization ----
depth_norm = cv2.normalize(
    depth, None, 0, 255, cv2.NORM_MINMAX
).astype(np.uint8)

depth_color = cv2.applyColorMap(
    depth_norm, cv2.COLORMAP_INFERNO
)

# OpenCV uses BGR → convert to RGB for matplotlib
depth_color_rgb = cv2.cvtColor(depth_color, cv2.COLOR_BGR2RGB)

# ---- display with matplotlib ----
plt.figure(figsize=(6, 6))
plt.imshow(depth_color_rgb)
plt.axis("off")
plt.title("Depth Anything V2 (Small)")
plt.show()